# Préparation des données

In [1]:
import numpy as np
import pandas as pd

print("NumPy", np.__version__, "| Pandas", pd.__version__)

NumPy 2.3.5 | Pandas 2.3.3


## 1. Génération des données simulées

In [2]:
rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    "id_etudiant": np.arange(1, n + 1),
    "age": rng.integers(19, 28, n),
    "sexe": rng.choice(["M", "F"], n),
    "filiere": rng.choice(["Informatique", "Mathématiques", "Physique", "Gestion"], n),
    "niveau": rng.choice(["L1", "L2", "L3"], n),
    "heures_etude": rng.normal(12, 5, n).clip(0, 40).round(1),
    "absences": rng.poisson(4, n),
    "internet": rng.choice(["Oui", "Non"], n, p=[0.8, 0.2]),
    "reseaux_sociaux": rng.normal(3, 1.5, n).clip(0, 10).round(1),
    "participation_td": rng.uniform(30, 100, n).round(0),
})

base = (6 + 0.25 * df["heures_etude"] - 0.3 * df["absences"]
        + 0.03 * df["participation_td"] - 0.2 * df["reseaux_sociaux"])
df["note_cc"] = (base + rng.normal(0, 1.5, n)).clip(0, 20).round(2)
df["note_examen"] = (base + rng.normal(0, 2.5, n)).clip(0, 20).round(2)
df["moyenne"] = (0.4 * df["note_cc"] + 0.6 * df["note_examen"]).round(2)
df["resultat"] = np.where(df["moyenne"] >= 10, "Réussi", "Échec")

# On "salit" un peu les données pour avoir quelque chose à nettoyer
for col in ["heures_etude", "note_cc", "internet", "age"]:
    df.loc[rng.choice(n, 15, replace=False), col] = np.nan
df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)

df.to_csv("../data/etudiants.csv", index=False)
print(df.shape)

(508, 14)


## 2. Import et exploration

In [3]:
df = pd.read_csv("../data/etudiants.csv")
df.head()

,id_etudiant,age,sexe,filiere,niveau,heures_etude,absences,internet,reseaux_sociaux,participation_td,note_cc,note_examen,moyenne,resultat
0,1,19.0,M,Gestion,L2,8.3,3,Oui,2.9,89.0,6.84,14.26,11.29,Réussi
1,2,25.0,F,Physique,L1,16.6,5,NaN,2.3,92.0,9.71,13.92,12.24,Réussi
2,3,24.0,F,Physique,L1,12.2,3,Oui,2.0,74.0,10.40,10.03,10.18,Réussi
3,4,22.0,F,Gestion,L1,10.6,5,Oui,5.3,74.0,9.24,7.77,8.36,Échec
4,5,22.0,M,Informatique,L2,11.5,3,Oui,2.9,73.0,10.24,11.03,10.71,Réussi


In [4]:
display(df.tail())
print("Observations :", df.shape[0])
print("Variables    :", df.shape[1])

,id_etudiant,age,sexe,filiere,niveau,heures_etude,absences,internet,reseaux_sociaux,participation_td,note_cc,note_examen,moyenne,resultat
503,68,22.0,M,Physique,L2,9.4,8,Oui,2.5,96.0,9.68,11.43,10.73,Réussi
504,480,19.0,F,Mathématiques,L3,23.6,5,Oui,1.4,42.0,15.05,16.68,16.03,Réussi
505,486,21.0,F,Gestion,L3,11.5,3,Oui,2.5,42.0,7.76,10.45,9.37,Échec
506,311,24.0,M,Physique,L1,20.1,3,Oui,3.6,82.0,12.76,12.23,12.44,Réussi
507,32,21.0,M,Informatique,L1,12.7,4,Oui,0.5,69.0,9.88,6.93,8.11,Échec


Observations : 508
Variables    : 14


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_etudiant       508 non-null    int64  
 1   age               493 non-null    float64
 2   sexe              508 non-null    object 
 3   filiere           508 non-null    object 
 4   niveau            508 non-null    object 
 5   heures_etude      492 non-null    float64
 6   absences          508 non-null    int64  
 7   internet          493 non-null    object 
 8   reseaux_sociaux   508 non-null    float64
 9   participation_td  508 non-null    float64
 10  note_cc           493 non-null    float64
 11  note_examen       508 non-null    float64
 12  moyenne           508 non-null    float64
 13  resultat          508 non-null    object 
dtypes: float64(7), int64(2), object(5)
memory usage: 55.7+ KB


In [6]:
df.describe()

,id_etudiant,age,heures_etude,absences,reseaux_sociaux,participation_td,note_cc,note_examen,moyenne
count,508.000000,493.000000,492.000000,508.000000,508.000000,508.000000,493.000000,508.000000,508.000000
mean,250.631890,23.014199,12.167886,4.000000,2.939567,63.279528,9.287383,9.243681,9.248819
std,145.013243,2.570510,4.993497,2.001971,1.444144,20.339636,2.278292,2.941645,2.258238
min,1.000000,19.000000,0.000000,0.000000,0.000000,30.000000,2.130000,0.820000,2.000000
25%,124.750000,21.000000,8.975000,3.000000,1.900000,45.000000,7.780000,7.440000,7.717500
50%,251.500000,23.000000,12.200000,4.000000,2.900000,61.000000,9.320000,9.385000,9.265000
75%,375.250000,25.000000,15.400000,5.000000,3.900000,81.000000,10.770000,11.125000,10.750000
max,500.000000,27.000000,26.000000,14.000000,7.300000,100.000000,17.630000,18.050000,16.410000


In [7]:
print(df.isna().sum())
print("Doublons :", df.duplicated().sum())

id_etudiant          0
age                 15
sexe                 0
filiere              0
niveau               0
heures_etude        16
absences             0
internet            15
reseaux_sociaux      0
participation_td     0
note_cc             15
note_examen          0
moyenne              0
resultat             0
dtype: int64
Doublons : 8


## 3. Suppression des doublons

In [8]:
avant = len(df)
df = df.drop_duplicates()
print("Lignes supprimées :", avant - len(df))
print("Nouvelle taille :", df.shape)

Lignes supprimées : 8
Nouvelle taille : (500, 14)


## 4. Traitement des valeurs manquantes

In [9]:
print(df.isna().sum())

id_etudiant          0
age                 15
sexe                 0
filiere              0
niveau               0
heures_etude        15
absences             0
internet            15
reseaux_sociaux      0
participation_td     0
note_cc             15
note_examen          0
moyenne              0
resultat             0
dtype: int64


In [10]:
for col in ["age", "heures_etude", "note_cc"]:
    df[col] = df[col].fillna(df[col].median())

df["internet"] = df["internet"].fillna(df["internet"].mode()[0])

print("Valeurs manquantes restantes :", df.isna().sum().sum())

Valeurs manquantes restantes : 0


## 5. Correction des types

In [11]:
df["age"] = df["age"].astype(int)
for col in ["sexe", "filiere", "niveau", "internet", "resultat"]:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 499
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   id_etudiant       500 non-null    int64   
 1   age               500 non-null    int64   
 2   sexe              500 non-null    category
 3   filiere           500 non-null    category
 4   niveau            500 non-null    category
 5   heures_etude      500 non-null    float64 
 6   absences          500 non-null    int64   
 7   internet          500 non-null    category
 8   reseaux_sociaux   500 non-null    float64 
 9   participation_td  500 non-null    float64 
 10  note_cc           500 non-null    float64 
 11  note_examen       500 non-null    float64 
 12  moyenne           500 non-null    float64 
 13  resultat          500 non-null    category
dtypes: category(5), float64(6), int64(3)
memory usage: 42.2 KB


## 6. Enregistrement de la base nettoyée

In [12]:
df.to_csv("../data/etudiants_propre.csv", index=False)
print("Fichier enregistré :", df.shape)

Fichier enregistré : (500, 14)


## Conclusion
La base initiale contenait 508 lignes. Après suppression de 8 doublons et remplacement des valeurs manquantes (médiane pour les variables numériques, valeur la plus fréquente pour `internet`), la base finale compte 500 observations et 14 variables. Elle est enregistrée dans `data/etudiants_propre.csv`.